Dans ce notebook on va merge différentes tables de données pour avoir un maximum de feature pouvant nous aider à prédire le prix au mètre carré d'un bien en Ile de France 

# Lib imports 

In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
print(os.getcwd())
os.chdir("../")
print(os.getcwd())

c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction\notebooks
c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction


# 1.Données géographiques, économiques et démographiques par code commune 

In [3]:
demographic_data = pd.read_excel("data/raw/data_par_commune/POPULATION_MUNICIPALE_COMMUNES_FRANCE.xlsx")
demographic_data.head()

,objectid,reg,dep,cv,codgeo,libgeo,p13_pop,p14_pop,p15_pop,p16_pop,p17_pop,p18_pop,p19_pop,p20_pop,p21_pop
0,115658,52,85,8502,85062,Châteauneuf,968.0,993.0,1013.0,1027.0,1056,1085.0,1114.0,1118.0,1134.0
1,115659,26,58,5808,58300,Urzy,1839.0,1835.0,1828.0,1802.0,1775,1749.0,1746.0,1747.0,1742.0
2,115660,43,70,7012,70137,Chassey-lès-Montbozon,218.0,217.0,216.0,215.0,217,215.0,215.0,220.0,225.0
3,115661,21,51,5123,51649,Vitry-le-François,13174.0,13144.0,12805.0,12552.0,12133,11743.0,11376.0,11458.0,11454.0
4,115662,11,78,7811,78638,Vaux-sur-Seine,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0


In [4]:
# reg = 11 c'est l'ile de France 
demographic_data = demographic_data[demographic_data["reg"] == 11]
demographic_data = demographic_data.rename(columns={"codgeo": "code_commune", 
                                                    'p13_pop': "pop_2016",
                                                    'p14_pop': "pop_2017",
                                                    'p15_pop': "pop_2018",
                                                    'p16_pop': "pop_2019",
                                                    'p17_pop': "pop_2020",
                                                    'p18_pop': "pop_2021",
                                                    'p19_pop': "pop_2022",
                                                    'p20_pop': "pop_2023",
                                                    'p21_pop': "pop_2024"})
demographic_data

,objectid,reg,dep,cv,code_commune,libgeo,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024
4,115662,11,78,7811,78638,Vaux-sur-Seine,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0
88,126398,11,77,7714,77295,Moisenay,1314.0,1333.0,1352.0,1371.0,1380,1383.0,1387.0,1379.0,1371.0
95,126405,11,95,9520,95301,Haute-Isle,301.0,290.0,279.0,279.0,278,282.0,286.0,290.0,290.0
109,126419,11,78,7807,78402,Mézières-sur-Seine,3626.0,3647.0,3636.0,3656.0,3676,3707.0,3683.0,3776.0,3826.0
193,126588,11,91,9105,91578,Saint-Sulpice-de-Favières,326.0,328.0,317.0,305.0,294,286.0,278.0,268.0,270.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34911,110172,11,77,7716,77181,Ferrières-en-Brie,2793.0,2801.0,3012.0,3222.0,3433,3483.0,3768.0,3796.0,3841.0
34946,110207,11,95,9517,95446,Nesles-la-Vallée,1799.0,1780.0,1801.0,1822.0,1843,1824.0,1803.0,1780.0,1803.0
34952,110213,11,95,9509,95056,Belloy-en-France,2115.0,2162.0,2165.0,2177.0,2189,2211.0,2203.0,2217.0,2228.0
34966,110227,11,91,9103,91570,Saint-Michel-sur-Orge,20057.0,19896.0,20160.0,19866.0,19758,19965.0,20484.0,21298.0,21437.0


Ajout du taux de croissance annuelle de la population :

In [5]:
# Calculer les taux de croissance annuels pour toutes les années de 2019 à 2024
for annee in range(2019, 2025):
    nb_annees = annee - 2016
    demographic_data[f'taux_croissance_pop_annuel_2016_{annee}'] = (
        (demographic_data[f'pop_{annee}'] / demographic_data['pop_2016']) ** (1/nb_annees) - 1
    ) * 100

demographic_data = demographic_data.drop(columns=['reg', 'dep', 'cv', 'libgeo', 'objectid'])
demographic_data

,code_commune,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024,taux_croissance_pop_annuel_2016_2019,taux_croissance_pop_annuel_2016_2020,taux_croissance_pop_annuel_2016_2021,taux_croissance_pop_annuel_2016_2022,taux_croissance_pop_annuel_2016_2023,taux_croissance_pop_annuel_2016_2024
4,78638,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0,0.752379,0.924149,0.746816,0.895685,0.795949,0.853214
88,77295,1314.0,1333.0,1352.0,1371.0,1380,1383.0,1387.0,1379.0,1371.0,1.425548,1.232726,1.028839,0.905193,0.692137,0.532217
95,95301,301.0,290.0,279.0,279.0,278,282.0,286.0,290.0,290.0,-2.498214,-1.967614,-1.295598,-0.848355,-0.530436,-0.464286
109,78402,3626.0,3647.0,3636.0,3656.0,3676,3707.0,3683.0,3776.0,3826.0,0.275029,0.342964,0.442834,0.260297,0.580753,0.673379
193,91578,326.0,328.0,317.0,305.0,294,286.0,278.0,268.0,270.0,-2.195070,-2.549868,-2.584136,-2.619680,-2.759919,-2.328407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34911,77181,2793.0,2801.0,3012.0,3222.0,3433,3483.0,3768.0,3796.0,3841.0,4.878113,5.293295,4.514491,5.117089,4.480795,4.063079
34946,95446,1799.0,1780.0,1801.0,1822.0,1843,1824.0,1803.0,1780.0,1803.0,0.424359,0.605921,0.276400,0.037023,-0.151565,0.027766
34952,95056,2115.0,2162.0,2165.0,2177.0,2189,2211.0,2203.0,2217.0,2228.0,0.967752,0.863457,0.891754,0.681734,0.675128,0.652740
34966,91570,20057.0,19896.0,20160.0,19866.0,19758,19965.0,20484.0,21298.0,21437.0,-0.318442,-0.374790,-0.091907,0.351715,0.861330,0.835221


In [6]:
economic_data = pd.read_parquet("data/raw/data_par_commune/base-comparateur-de-territoires.parquet")
economic_data = economic_data.replace("s", pd.NA) # la valeur "s" signifie généralement que la donnée est confidentielle ou non disponible pour des raisons statistiques.
economic_data

,codgeo,nom_officiel_commune_arrondissement_municipal,p20_pop,p14_pop,superf,nais1420,dece1420,p20_men,naisd22,decesd22,...,etbe21,etfz21,etgu21,etgz21,etoq21,ettef121,ettefp1021,idf,geo_point,geo_shape
0,77121,Collégien,3339,3329,4.27,221,73,1258.000000,29,16,...,44,54,249,127,5,218,119,Oui,b'\x01\x01\x00\x00\x00\xe3\xf0U\xfa\xb6j\x05@\...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x004\x00\x0...
1,77249,Lésigny,7125,7387,10.13,331,243,2754.688016,53,34,...,3,21,108,26,19,127,13,Oui,b'\x01\x01\x00\x00\x00\x1eP\x9ct}\xed\x04@\xb7...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x81\x00...
2,77273,Marchémoret,594,557,7.04,24,8,202.907726,8,4,...,0,3,5,1,2,11,0,Oui,b'\x01\x01\x00\x00\x00W\xfb\xf2Y\x9c\x03\x06@G...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00N\x00\x0...
3,78010,Les Alluets-le-Roi,1215,1237,7.39,56,44,464.300630,12,6,...,1,15,37,17,5,50,9,Oui,b'\x01\x01\x00\x00\x000\xcb\x94\xff\xfe\xa3\xf...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00i\x00\x0...
4,78036,Autouillet,612,470,4.93,29,14,227.601970,7,1,...,1,0,2,0,2,5,0,Oui,b'\x01\x01\x00\x00\x00\xbe%\xdfc\x86\xc0\xfc?\...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00C\x00\x0...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,91645,Verrières-le-Buisson,14602,15711,9.91,733,969,6091.947452,125,196,...,25,55,291,80,43,301,86,Oui,b'\x01\x01\x00\x00\x00\xd7\xb0YKO\x04\x02@j$\x...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x9e\x00...
1283,94079,Villiers-sur-Marne,29672,28278,4.33,2756,1080,12555.052456,552,194,...,22,125,418,107,59,492,76,Oui,b'\x01\x01\x00\x00\x00[\x95\xcfv\xbd\\\x04@\xf...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00O\x00\x0...
1284,95039,Auvers-sur-Oise,6792,6943,12.69,406,251,2876.344755,61,47,...,8,29,104,21,9,122,12,Oui,b'\x01\x01\x00\x00\x00z\xb9L\x7f\xc1<\x01@| (@...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00z\x00\x0...
1285,95042,Baillet-en-France,1893,2031,7.91,121,64,759.041633,20,10,...,10,19,47,17,5,62,16,Oui,b'\x01\x01\x00\x00\x00!\xe4\x17kBk\x02@\x86\xb...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00o\x00\x0...


In [7]:
economic_data = economic_data[['codgeo', 'superf', 'p20_men', 'p20_log', 'p20_rp', 'p20_rsecocc', 'p20_logvac',
       'p20_rp_prop', 'nbmenfisc20', 'med20', 'tp6020', 'p20_emplt',
       'p20_emplt_sal', 'p14_emplt', 'p20_pop1564', 'p20_chom1564',
       'p20_act1564', 'ettot21', 'etaz21', 'etbe21', 'etfz21', 'etgu21',
       'etgz21', 'etoq21', 'ettef121', 'ettefp1021']]

In [8]:
rename_cols = {
    'codgeo': 'code_commune',
    'superf': 'superficie_commune_m2',
    'p20_men': 'nb_menages_2020',
    'p20_log': 'nb_logements_total_2020',
    'p20_rp': 'nb_residences_principales_2020',
    'p20_rsecocc': 'nb_residences_secondaires_2020',
    'p20_logvac': 'nb_logements_vacants_2020',
    'p20_rp_prop': 'nb_residences_principales_proprietaires_2020',
    'med20': 'revenu_median_2020',
    'tp6020': 'taux_pauvrete_60pct_2020',
    'p20_emplt': 'emploi_total_2020',
    'p20_emplt_sal': 'emploi_salarie_2020',
    'p14_emplt': 'emploi_total_2014',
    'p20_pop1564': 'population_15_64_2020',
    'p20_chom1564': 'chomeurs_15_64_2020',
    'p20_act1564': 'nb_actifs_15_64_2020',
    'ettot21': 'etablissements_total_2021',
    'etaz21': 'etablissements_agriculture_2021',
    'etbe21': 'etablissements_industrie_2021',
    'etfz21': 'etablissements_construction_2021',
    'etgu21': 'etablissements_commerce_transport_2021',
    'etgz21': 'etablissements_services_entreprises_2021',
    'etoq21': 'etablissements_services_publics_sante_2021',
    'ettef121': 'etablissements_1_salarie_2021',
    'ettefp1021': 'etablissements_10_plus_salaries_2021',
    'nbmenfisc20': 'nb_menages_fiscaux_2020'
}

# Renommage
economic_data = economic_data.rename(columns=rename_cols)
economic_data

,code_commune,superficie_commune_m2,nb_menages_2020,nb_logements_total_2020,nb_residences_principales_2020,nb_residences_secondaires_2020,nb_logements_vacants_2020,nb_residences_principales_proprietaires_2020,nb_menages_fiscaux_2020,revenu_median_2020,...,nb_actifs_15_64_2020,etablissements_total_2021,etablissements_agriculture_2021,etablissements_industrie_2021,etablissements_construction_2021,etablissements_commerce_transport_2021,etablissements_services_entreprises_2021,etablissements_services_publics_sante_2021,etablissements_1_salarie_2021,etablissements_10_plus_salaries_2021
0,77121,4.27,1258.000000,1318.000000,1258.000000,18.000000,42.000000,845.000000,1242,25710,...,1694.000000,352,0,44,54,249,127,5,218,119
1,77249,10.13,2754.688016,2881.952676,2754.688016,24.050014,103.214645,2357.264137,2656,30460,...,3360.279613,153,2,3,21,108,26,19,127,13
2,77273,7.04,202.907726,215.562668,202.907726,2.920371,9.734570,167.480151,194,26550,...,342.333401,11,1,0,3,5,1,2,11,0
3,78010,7.39,464.300630,513.862381,464.300630,10.325365,39.236387,422.804167,478,35110,...,551.251291,63,5,1,15,37,17,5,50,9
4,78036,4.93,227.601970,264.167010,227.601970,24.028455,12.536585,209.978713,218,31960,...,309.157803,6,1,1,0,2,0,2,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,91645,9.91,6091.947452,6565.061687,6091.947452,84.005691,389.108544,4393.282448,6227,35320,...,6363.848234,415,1,25,55,291,80,43,301,86
1283,94079,4.33,12555.052456,13288.446832,12555.052456,214.678221,518.716156,6385.956398,12878,23450,...,14828.293436,624,0,22,125,418,107,59,492,76
1284,95039,12.69,2876.344755,3138.832196,2876.344755,83.154419,179.333023,2236.154737,2853,28110,...,3347.074629,152,2,8,29,104,21,9,122,12
1285,95042,7.91,759.041633,799.142515,759.041633,4.010088,36.090794,648.178129,734,29430,...,991.564956,82,1,10,19,47,17,5,62,16


In [9]:
economic_data.isnull().mean() * 100

code_commune                                     0.000000
superficie_commune_m2                            0.000000
nb_menages_2020                                  0.000000
nb_logements_total_2020                          0.000000
nb_residences_principales_2020                   0.000000
nb_residences_secondaires_2020                   0.000000
nb_logements_vacants_2020                        0.000000
nb_residences_principales_proprietaires_2020     0.000000
nb_menages_fiscaux_2020                          1.243201
revenu_median_2020                               1.243201
taux_pauvrete_60pct_2020                        62.859363
emploi_total_2020                                0.000000
emploi_salarie_2020                              0.000000
emploi_total_2014                                0.000000
population_15_64_2020                            0.000000
chomeurs_15_64_2020                              0.000000
nb_actifs_15_64_2020                             0.000000
etablissements

On drop taux de pauvreté car trop de valeurs manquantes et cela sera surement capté par d'autres variables de revenus :

In [10]:
economic_data =economic_data.drop(columns=['taux_pauvrete_60pct_2020'])

Créer de nouveaux indicateurs :

In [11]:
# Densité de population (hab/km²)
economic_data["densite_population_active_2020"] = economic_data["population_15_64_2020"] / (economic_data["superficie_commune_m2"] / 1_000_000)

# Structure du logement
economic_data["taux_residences_secondaires"] = economic_data["nb_residences_secondaires_2020"] / economic_data["nb_logements_total_2020"]
economic_data["taux_logements_vacants"] = economic_data["nb_logements_vacants_2020"] / economic_data["nb_logements_total_2020"]
economic_data["taux_proprietaires"] = economic_data["nb_residences_principales_proprietaires_2020"] / economic_data["nb_residences_principales_2020"]
economic_data["ratio_residences_secondaires_population"] = economic_data["nb_residences_secondaires_2020"] / economic_data["population_15_64_2020"]
economic_data['densite_residences_principales_2020'] = (economic_data['nb_residences_principales_2020'] / (economic_data['superficie_commune_m2'] / 1_000_000)) # en km²

# Emploi & chômage
economic_data["taux_chomage_15_64"] = economic_data["chomeurs_15_64_2020"] / economic_data["nb_actifs_15_64_2020"]
economic_data["taux_emploi"] = economic_data["emploi_total_2020"] / economic_data["population_15_64_2020"]
economic_data["evolution_emploi_2014_2020"] = (economic_data["emploi_total_2020"] - economic_data["emploi_total_2014"]) / economic_data["emploi_total_2014"]

# Activité économique
economic_data["part_services_entreprises"] = economic_data["etablissements_services_entreprises_2021"] / economic_data["etablissements_total_2021"]
economic_data["part_commerce_tourisme"] = economic_data["etablissements_commerce_transport_2021"] / economic_data["etablissements_total_2021"]
economic_data["part_admin_sante"] = economic_data["etablissements_services_publics_sante_2021"] / economic_data["etablissements_total_2021"]

# Tissu économique local
economic_data["etablissements_par_menage"] = economic_data["etablissements_total_2021"] / economic_data["nb_menages_2020"]
economic_data["taux_etablissements_10_plus"] = economic_data["etablissements_10_plus_salaries_2021"] / economic_data["etablissements_total_2021"]

economic_data

,code_commune,superficie_commune_m2,nb_menages_2020,nb_logements_total_2020,nb_residences_principales_2020,nb_residences_secondaires_2020,nb_logements_vacants_2020,nb_residences_principales_proprietaires_2020,nb_menages_fiscaux_2020,revenu_median_2020,...,ratio_residences_secondaires_population,densite_residences_principales_2020,taux_chomage_15_64,taux_emploi,evolution_emploi_2014_2020,part_services_entreprises,part_commerce_tourisme,part_admin_sante,etablissements_par_menage,taux_etablissements_10_plus
0,77121,4.27,1258.000000,1318.000000,1258.000000,18.000000,42.000000,845.000000,1242,25710,...,0.008291,2.946136e+08,0.083825,1.485098,0.173768,0.360795,0.707386,0.014205,0.279809,0.338068
1,77249,10.13,2754.688016,2881.952676,2754.688016,24.050014,103.214645,2357.264137,2656,30460,...,0.005474,2.719337e+08,0.076722,0.217478,0.022532,0.169935,0.705882,0.124183,0.055542,0.084967
2,77273,7.04,202.907726,215.562668,202.907726,2.920371,9.734570,167.480151,194,26550,...,0.007175,2.882212e+07,0.058589,0.104832,0.138353,0.090909,0.454545,0.181818,0.054212,0.000000
3,78010,7.39,464.300630,513.862381,464.300630,10.325365,39.236387,422.804167,478,35110,...,0.013537,6.282823e+07,0.075026,0.496032,0.111444,0.269841,0.587302,0.079365,0.135688,0.142857
4,78036,4.93,227.601970,264.167010,227.601970,24.028455,12.536585,209.978713,218,31960,...,0.059716,4.616673e+07,0.060957,0.086112,-0.008080,0.000000,0.333333,0.333333,0.026362,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,91645,9.91,6091.947452,6565.061687,6091.947452,84.005691,389.108544,4393.282448,6227,35320,...,0.009830,6.147273e+08,0.085815,0.403862,-0.113901,0.192771,0.701205,0.103614,0.068123,0.207229
1283,94079,4.33,12555.052456,13288.446832,12555.052456,214.678221,518.716156,6385.956398,12878,23450,...,0.010882,2.899550e+09,0.090059,0.274468,0.036547,0.171474,0.669872,0.094551,0.049701,0.121795
1284,95039,12.69,2876.344755,3138.832196,2876.344755,83.154419,179.333023,2236.154737,2853,28110,...,0.019182,2.266623e+08,0.080417,0.209048,-0.140841,0.138158,0.684211,0.059211,0.052845,0.078947
1285,95042,7.91,759.041633,799.142515,759.041633,4.010088,36.090794,648.178129,734,29430,...,0.003248,9.595975e+07,0.070349,0.434082,-0.027186,0.207317,0.573171,0.060976,0.108031,0.195122


In [12]:
economic_data = economic_data.drop(columns=["emploi_total_2014",
                                            "emploi_total_2020",
                                            "chomeurs_15_64_2020",
                                            "nb_residences_secondaires_2020",
                                            "nb_logements_total_2020",
                                            "nb_logements_vacants_2020", 
                                            "nb_residences_principales_proprietaires_2020",
                                            "etablissements_services_entreprises_2021",
                                            "etablissements_commerce_transport_2021",
                                            "etablissements_services_publics_sante_2021",
                                            "etablissements_total_2021",
                                            "nb_menages_2020",
                                            "nb_residences_principales_2020", 
                                            "nb_menages_fiscaux_2020",
                                            "population_15_64_2020",
                                            "nb_actifs_15_64_2020", 
                                            "emploi_salarie_2020"
                                            ])

In [13]:
economic_data.columns

Index(['code_commune', 'superficie_commune_m2', 'revenu_median_2020',
       'etablissements_agriculture_2021', 'etablissements_industrie_2021',
       'etablissements_construction_2021', 'etablissements_1_salarie_2021',
       'etablissements_10_plus_salaries_2021',
       'densite_population_active_2020', 'taux_residences_secondaires',
       'taux_logements_vacants', 'taux_proprietaires',
       'ratio_residences_secondaires_population',
       'densite_residences_principales_2020', 'taux_chomage_15_64',
       'taux_emploi', 'evolution_emploi_2014_2020',
       'part_services_entreprises', 'part_commerce_tourisme',
       'part_admin_sante', 'etablissements_par_menage',
       'taux_etablissements_10_plus'],
      dtype='object')

Données de score de developemment humain par commune :

In [14]:
idh_data = pd.read_parquet("data/raw/data_par_commune/indice-de-developpement-humain-idh2-des-communes-dile-de-france.parquet")
idh_data

,objectid,insee,annee,sante_plaf,educ_plaf,revenu_plaf,idh2,nomcom,pop
0,238,77227,2013,0.720523,0.345290,0.484803,0.516872,Hermé,646
1,345,77336,2013,0.683653,0.516738,0.595381,0.598591,Neufmoutiers-en-Brie,921
2,354,77345,2013,0.480528,0.427034,0.540652,0.482738,Orly-sur-Morin,676
3,359,77352,2013,0.648913,0.495740,0.604881,0.583178,Ozouer-le-Voulgis,1837
4,447,77443,2013,0.414469,0.465897,0.557855,0.479407,Sancy,379
...,...,...,...,...,...,...,...,...,...
3892,3165,78090,1999,0.388818,0.178893,0.537241,0.368318,Bouafle,2014
3893,3194,78171,1999,0.713465,0.356684,0.572949,0.547699,Condé-sur-Vesgre,1043
3894,3635,93027,1999,0.384332,0.083452,0.077246,0.181677,La Courneuve,35301
3895,3802,95308,1999,0.660175,0.344287,0.587841,0.530768,Hérouville,598


In [15]:
rename_dict = {
    'insee': 'code_commune',
    'sante_plaf': 'sante_score_2013_commune',
    'educ_plaf': 'education_score_2013_commune',
    'revenu_plaf': 'revenu_score_2013_commune',
    'idh2': 'idh2_2013_commune'
}
idh_data = idh_data.rename(columns=rename_dict)
idh_data = idh_data.drop(columns=['objectid', 'nomcom', 'pop'])
idh_data_2013 = idh_data[idh_data['annee'] == "2013"].drop(columns=['annee']).reset_index(drop=True)
idh_data_2013

,code_commune,sante_score_2013_commune,education_score_2013_commune,revenu_score_2013_commune,idh2_2013_commune
0,77227,0.720523,0.345290,0.484803,0.516872
1,77336,0.683653,0.516738,0.595381,0.598591
2,77345,0.480528,0.427034,0.540652,0.482738
3,77352,0.648913,0.495740,0.604881,0.583178
4,77443,0.414469,0.465897,0.557855,0.479407
...,...,...,...,...,...
1294,91044,0.774009,0.619009,0.662485,0.685168
1295,91411,0.750100,0.712545,0.758648,0.740431
1296,91587,0.616425,0.561824,0.616408,0.598219
1297,95051,0.744042,0.506516,0.626639,0.625732


Données sur la criminalité par commune :

In [16]:
crimes_data = pd.read_parquet("data/raw/data_par_commune/donnee-comm-data.gouv-parquet-2024-geographie2025-produit-le2025-06-04.parquet")
crimes_data = crimes_data.rename(columns={'CODGEO_2025': 'code_commune'})
crimes_data

,code_commune,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,ndiff,767,2016,348,2016,1.362069,0.959839
2,01001,2016,Violences sexuelles,Victime,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
3,01001,2016,Vols avec armes,Infraction,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.000000,diff,767,2016,348,2016,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4714195,97617,2024,Destructions et dégradations volontaires,Infraction,93.0,6.674322,diff,13934,2017,3872,2017,NaN,NaN
4714196,97617,2024,Usage de stupéfiants,Mis en cause,NaN,NaN,ndiff,13934,2017,3872,2017,7.454546,0.874061
4714197,97617,2024,Usage de stupéfiants (AFD),Mis en cause,NaN,NaN,ndiff,13934,2017,3872,2017,8.714286,0.903075
4714198,97617,2024,Trafic de stupéfiants,Mis en cause,NaN,NaN,ndiff,13934,2017,3872,2017,3.357143,0.270960


In [17]:
# Calcul du total et du taux moyen par commune sur toutes les années
crime_rate_data = crimes_data.groupby('code_commune').agg({
    'nombre': 'sum',  # total des crimes
    'insee_pop': 'mean'  # population moyenne sur les années
}).reset_index()

crime_rate_data['taux_criminalite_moyen'] = crime_rate_data['nombre'] / crime_rate_data['insee_pop']

crime_rate_data = crime_rate_data[['code_commune', 'taux_criminalite_moyen']]
crime_rate_data

,code_commune,taux_criminalite_moyen
0,01001,0.000000
1,01002,0.000000
2,01004,0.443338
3,01005,0.095197
4,01006,0.000000
...,...,...
34915,97613,0.154073
34916,97614,0.222190
34917,97615,0.386471
34918,97616,0.233955


Dataframe final d'indicateurs communaux

In [18]:
for df in [demographic_data, economic_data, idh_data_2013, crime_rate_data]:
    df['code_commune'] = df['code_commune'].astype(str)
    
# Rassembler tous les dataframes en un seul
data_par_commune = demographic_data.merge(economic_data, on='code_commune', how='left') \
                           .merge(idh_data_2013, on='code_commune', how='left') \
                           .merge(crime_rate_data, on='code_commune', how='left')

data_par_commune

,code_commune,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024,...,part_services_entreprises,part_commerce_tourisme,part_admin_sante,etablissements_par_menage,taux_etablissements_10_plus,sante_score_2013_commune,education_score_2013_commune,revenu_score_2013_commune,idh2_2013_commune,taux_criminalite_moyen
0,78638,4749.0,4715.0,4788.0,4857.0,4927,4929.0,5010.0,5020.0,5083.0,...,0.159574,0.723404,0.085106,0.047612,0.063830,0.486201,0.558575,0.626984,0.557253,0.258535
1,77295,1314.0,1333.0,1352.0,1371.0,1380,1383.0,1387.0,1379.0,1371.0,...,0.086957,0.347826,0.217391,0.043502,0.130435,0.595452,0.458679,0.606847,0.553659,0.053883
2,95301,301.0,290.0,279.0,279.0,278,282.0,286.0,290.0,290.0,...,0.500000,0.500000,0.500000,0.014925,0.000000,0.607826,0.530290,0.609074,0.582397,0.000000
3,78402,3626.0,3647.0,3636.0,3656.0,3676,3707.0,3683.0,3776.0,3826.0,...,0.210526,0.500000,0.144737,0.051287,0.144737,0.655672,0.472022,0.589140,0.572278,0.180551
4,91578,326.0,328.0,317.0,305.0,294,286.0,278.0,268.0,270.0,...,0.100000,0.700000,0.200000,0.080645,0.100000,0.893288,0.600301,0.717455,0.737015,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,77181,2793.0,2801.0,3012.0,3222.0,3433,3483.0,3768.0,3796.0,3841.0,...,0.231788,0.695364,0.052980,0.093617,0.410596,0.653765,0.587293,0.627911,0.622990,0.386791
1283,95446,1799.0,1780.0,1801.0,1822.0,1843,1824.0,1803.0,1780.0,1803.0,...,0.180000,0.620000,0.120000,0.064908,0.080000,0.674299,0.628949,0.678839,0.660696,0.157489
1284,95056,2115.0,2162.0,2165.0,2177.0,2189,2211.0,2203.0,2217.0,2228.0,...,0.245283,0.452830,0.113208,0.064744,0.132075,0.644696,0.482658,0.600748,0.576034,0.223398
1285,91570,20057.0,19896.0,20160.0,19866.0,19758,19965.0,20484.0,21298.0,21437.0,...,0.160000,0.597778,0.124444,0.050965,0.162222,0.663929,0.521276,0.503936,0.563047,0.384172


Densité de population par année :

In [19]:
for year in range(2016, 2025):
    pop_col = f'pop_{year}'
    density_col = f'densite_pop_{year}'
    # Calcul : population / superficie (en km² pour plus de lisibilité)
    data_par_commune[density_col] = (
        data_par_commune[pop_col] / (data_par_commune['superficie_commune_m2'] / 1_000_000)
    )

In [20]:
data_par_commune = data_par_commune.drop(columns=['pop_2016', 'pop_2017', 'pop_2018', 'pop_2019',
       'pop_2020', 'pop_2021', 'pop_2022', 'pop_2023', 'pop_2024', 'superficie_commune_m2'])

In [21]:
data_par_commune.isnull().mean() * 100

code_commune                               0.000000
taux_croissance_pop_annuel_2016_2019       0.000000
taux_croissance_pop_annuel_2016_2020       0.000000
taux_croissance_pop_annuel_2016_2021       0.000000
taux_croissance_pop_annuel_2016_2022       0.000000
taux_croissance_pop_annuel_2016_2023       0.000000
taux_croissance_pop_annuel_2016_2024       0.000000
revenu_median_2020                         1.243201
etablissements_agriculture_2021            0.000000
etablissements_industrie_2021              0.000000
etablissements_construction_2021           0.000000
etablissements_1_salarie_2021              0.000000
etablissements_10_plus_salaries_2021       0.000000
densite_population_active_2020             0.000000
taux_residences_secondaires                0.000000
taux_logements_vacants                     0.000000
taux_proprietaires                         0.000000
ratio_residences_secondaires_population    0.000000
densite_residences_principales_2020        0.000000
taux_chomage

Il y a des données manquantes qui pourraient être retrouvées car il y a des changements de code insee. 

Pour identifier facilement les données provenant de cette table on va renommber les variable avec le prefixe commune_

In [22]:
for col in data_par_commune.columns:
    if col != "code_commune":
        data_par_commune = data_par_commune.rename(columns={col : f"commune_{col}"})

data_par_commune

,code_commune,commune_taux_croissance_pop_annuel_2016_2019,commune_taux_croissance_pop_annuel_2016_2020,commune_taux_croissance_pop_annuel_2016_2021,commune_taux_croissance_pop_annuel_2016_2022,commune_taux_croissance_pop_annuel_2016_2023,commune_taux_croissance_pop_annuel_2016_2024,commune_revenu_median_2020,commune_etablissements_agriculture_2021,commune_etablissements_industrie_2021,...,commune_taux_criminalite_moyen,commune_densite_pop_2016,commune_densite_pop_2017,commune_densite_pop_2018,commune_densite_pop_2019,commune_densite_pop_2020,commune_densite_pop_2021,commune_densite_pop_2022,commune_densite_pop_2023,commune_densite_pop_2024
0,78638,0.752379,0.924149,0.746816,0.895685,0.795949,0.853214,28210,0,3,...,0.258535,5.620118e+08,5.579882e+08,5.666272e+08,5.747929e+08,5.830769e+08,5.833136e+08,5.928994e+08,5.940828e+08,6.015385e+08
1,77295,1.425548,1.232726,1.028839,0.905193,0.692137,0.532217,28610,1,1,...,0.053883,1.506881e+08,1.528670e+08,1.550459e+08,1.572248e+08,1.582569e+08,1.586009e+08,1.590596e+08,1.581422e+08,1.572248e+08
2,95301,-2.498214,-1.967614,-1.295598,-0.848355,-0.530436,-0.464286,24820,0,0,...,0.000000,1.171206e+08,1.128405e+08,1.085603e+08,1.085603e+08,1.081712e+08,1.097276e+08,1.112840e+08,1.128405e+08,1.128405e+08
3,78402,0.275029,0.342964,0.442834,0.260297,0.580753,0.673379,26560,2,3,...,0.180551,3.479846e+08,3.500000e+08,3.489443e+08,3.508637e+08,3.527831e+08,3.557582e+08,3.534549e+08,3.623800e+08,3.671785e+08
4,91578,-2.195070,-2.549868,-2.584136,-2.619680,-2.759919,-2.328407,35220,1,0,...,0.000000,7.459954e+07,7.505721e+07,7.254005e+07,6.979405e+07,6.727689e+07,6.544622e+07,6.361556e+07,6.132723e+07,6.178490e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,77181,4.878113,5.293295,4.514491,5.117089,4.480795,4.063079,28250,1,12,...,0.386791,4.137778e+08,4.149630e+08,4.462222e+08,4.773333e+08,5.085926e+08,5.160000e+08,5.582222e+08,5.623704e+08,5.690370e+08
1283,95446,0.424359,0.605921,0.276400,0.037023,-0.151565,0.027766,29650,3,4,...,0.157489,1.336553e+08,1.322437e+08,1.338039e+08,1.353640e+08,1.369242e+08,1.355126e+08,1.339525e+08,1.322437e+08,1.339525e+08
1284,95056,0.967752,0.863457,0.891754,0.681734,0.675128,0.652740,27310,2,7,...,0.223398,2.228662e+08,2.278188e+08,2.281349e+08,2.293994e+08,2.306639e+08,2.329821e+08,2.321391e+08,2.336143e+08,2.347734e+08
1285,91570,-0.318442,-0.374790,-0.091907,0.351715,0.861330,0.835221,22570,1,29,...,0.384172,3.791493e+09,3.761059e+09,3.810964e+09,3.755388e+09,3.734972e+09,3.774102e+09,3.872212e+09,4.026087e+09,4.052363e+09


# 2. Données des ventes et de leurs caractéristiques  

In [ ]:
# Charger et concaténer
years = ['2020', '2021', '2022', '2023', '2024', '2025']
df_vf = pd.concat([
    pd.read_csv(f"data/raw/initial_data_files/DVF_{year}.csv", sep=",", low_memory=False)
    for year in years
], ignore_index=True)

# Filtrer Île-de-France
departements_idf = ['75', '77', '78', '91', '92', '93', '94', '95']
raw_idf_data = df_vf[df_vf['code_departement'].isin(departements_idf)].copy()

# libérer mémoire 
del df_vf

# Supprimer colonnes inutiles (revoir plus tard si on peut en réutiliser certaines)
colonnes_a_supprimer = [
    'numero_disposition', 'adresse_suffixe', 'adresse_code_voie',
    'ancien_code_commune', 'ancien_nom_commune', 'ancien_id_parcelle',
    'numero_volume', 'code_nature_culture', 'code_nature_culture_speciale', 
    'id_mutation', 'adresse_numero', 'id_parcelle',
    'lot1_numero', 'lot2_numero', 'lot3_numero',
    'lot4_numero', 'lot5_numero', 'lot1_surface_carrez',
    'lot2_surface_carrez', 'lot3_surface_carrez', 'lot4_surface_carrez', 
    'lot5_surface_carrez', 'nature_culture', 'nature_culture_speciale',
    ]

raw_idf_data.drop(columns=colonnes_a_supprimer, inplace=True)
raw_idf_data.head()

Convertir date et créer variables temporelles

In [ ]:
raw_idf_data['date_mutation'] = pd.to_datetime(raw_idf_data['date_mutation'], format='%Y-%m-%d', errors='coerce')
raw_idf_data['annee'] = raw_idf_data['date_mutation'].dt.year
raw_idf_data['semestre'] = raw_idf_data['date_mutation'].dt.year.astype(str) + '-S' + np.where(raw_idf_data['date_mutation'].dt.quarter.gt(2), 2, 1).astype(str)
raw_idf_data['trimestre'] = raw_idf_data['date_mutation'].dt.year.astype(str) + '-T' + raw_idf_data['date_mutation'].dt.quarter.astype(str)

# Calculer le semestre précédent (S-1)
date_s_minus_1 = raw_idf_data['date_mutation'] - pd.DateOffset(months=6)
semestre_precedent = np.where(date_s_minus_1.dt.quarter.gt(2), 2, 1)
raw_idf_data['semestre_precedent'] = date_s_minus_1.dt.year.astype(str) + '-S' + semestre_precedent.astype(str)

# Calculer l'année précédente
raw_idf_data['annee_precedente'] = (raw_idf_data['date_mutation'] - pd.DateOffset(years=1)).dt.year

# Calculer le trimestre précédent
date_t_minus_1 = raw_idf_data['date_mutation'] - pd.DateOffset(months=3)
raw_idf_data['trimestre_precedent'] = date_t_minus_1.dt.year.astype(str) + 'T' + date_t_minus_1.dt.quarter.astype(str)

raw_idf_data[['date_mutation', "semestre", "trimestre", "semestre_precedent", "annee_precedente", "trimestre_precedent"]]

,date_mutation,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent
1569723,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569724,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569725,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569726,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
1569727,2020-07-01,2020-S2,2020-T3,2020-S1,2019,2020T2
...,...,...,...,...,...,...
20102734,2025-06-27,2025-S1,2025-T2,2024-S2,2024,2025T1
20102735,2025-06-27,2025-S1,2025-T2,2024-S2,2024,2025T1
20102736,2025-06-27,2025-S1,2025-T2,2024-S2,2024,2025T1
20102737,2025-06-25,2025-S1,2025-T2,2024-S2,2024,2025T1


Conversions numériques

In [ ]:
numeric_cols = ['valeur_fonciere', 'surface_reelle_bati', 'surface_terrain',
                'nombre_pieces_principales', 'nombre_lots']
for col in numeric_cols:
    if col in raw_idf_data.columns:
        if raw_idf_data[col].dtype == 'object':
            raw_idf_data[col] = raw_idf_data[col].str.replace(',', '.').astype(float)
        else:
            raw_idf_data[col] = pd.to_numeric(raw_idf_data[col], errors='coerce')

Filtres qualité, éliminer des outliers  

In [ ]:
raw_idf_data = raw_idf_data[
    (raw_idf_data['valeur_fonciere'] > 1000) &
    (raw_idf_data['surface_reelle_bati'] > 9) &
    (raw_idf_data['surface_reelle_bati'] < 1000) &
    (raw_idf_data['nombre_pieces_principales'] >= 1) &
    (raw_idf_data['nombre_pieces_principales'] <= 15)
].copy()

Créer la variable cible et enlever les ventes improbable (basé sur une étude de pertinence et des prix du marché)

In [ ]:
raw_idf_data['prix_m2'] = raw_idf_data['valeur_fonciere'] / raw_idf_data['surface_reelle_bati']

# Filtrer prix au m² aberrants
raw_idf_data = raw_idf_data[
    (raw_idf_data['prix_m2'] > 500) &
    (raw_idf_data['prix_m2'] < 20000)
].copy()

Prix médian par code commune et par semestre précédant la vente 

In [ ]:
prix_par_quartier_semestre = raw_idf_data.groupby(['code_commune', 'semestre'], observed=False)['prix_m2'].agg([
    ('commune_prix_median_m2', 'median'),
    ('commune_prix_moyen_m2', 'mean'),
    ('nb_transactions', 'count'),
]).reset_index()

prix_par_quartier_semestre

,code_commune,semestre,commune_prix_median_m2,commune_prix_moyen_m2,nb_transactions
0,75101,2020-S2,12258.730159,11782.522899,133
1,75101,2021-S1,13138.461538,13151.370658,119
2,75101,2021-S2,13208.719626,13045.431128,169
3,75101,2022-S1,12857.142857,12794.000198,175
4,75101,2022-S2,12833.333333,12721.461795,185
...,...,...,...,...,...
12169,95690,2023-S1,2701.658768,2701.658768,2
12170,95690,2023-S2,2708.433117,2708.433117,2
12171,95690,2024-S1,2131.578947,2131.578947,1
12172,95690,2024-S2,2575.301205,2685.291750,6


Filtrer les communes avec au moins 15 transactions pour obtenir une estimation un minimum robuste


In [ ]:
prix_par_quartier_semestre[(prix_par_quartier_semestre['nb_transactions'] <= 5)]

,code_commune,semestre,commune_prix_median_m2,commune_prix_moyen_m2,nb_transactions
209,77001,2025-S1,2892.222222,2744.693767,5
220,77003,2020-S2,1678.571429,1678.571429,1
221,77003,2021-S1,2980.000000,2568.333333,3
223,77003,2022-S1,2499.382716,2499.382716,2
224,77003,2022-S2,8380.353175,8380.353175,2
...,...,...,...,...,...
12167,95690,2022-S1,2768.421053,2768.421053,1
12169,95690,2023-S1,2701.658768,2701.658768,2
12170,95690,2023-S2,2708.433117,2708.433117,2
12171,95690,2024-S1,2131.578947,2131.578947,1


Là on a un problème sur les communes sur lesquelles on n'a pas de robustesse statistique. On pourrait mettre ces données en manquantes et faire un algorithme de plus proches voisins par exemple pour remplir ces valeurs. Pour l'instant on garde comme ça. 

Joindre le prix médian du mètre carré de la commune au dernier semestre au dataframe principal

In [ ]:
raw_idf_data = raw_idf_data.merge(
    prix_par_quartier_semestre[['code_commune', 'semestre', 'commune_prix_median_m2']],
    left_on=['code_commune', 'semestre_precedent'],
    right_on=['code_commune', 'semestre'],
    how='left'
)

In [ ]:
raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,...,latitude,annee,semestre_x,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,semestre_y,commune_prix_median_m2
0,2020-07-01,Vente,246600.0,SEN DES LONGUES RAIES,77670.0,77419,Saint-Mammès,77,0,1.0,...,48.383233,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2348.571429,NaN,NaN
1,2020-07-01,Vente,424000.0,RUE COROT,77310.0,77040,Boissise-le-Roi,77,0,1.0,...,48.529151,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN
2,2020-07-01,Vente,424000.0,RUE COROT,77310.0,77040,Boissise-le-Roi,77,0,1.0,...,48.529151,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,3164.179104,NaN,NaN
3,2020-07-01,Vente,192000.0,RUE DU VERT BUISSON,77550.0,77296,Moissy-Cramayel,77,5,1.0,...,48.627185,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,2206.896552,NaN,NaN
4,2020-07-02,Vente,160000.0,RUE DE LA MAIRIE,77123.0,77471,Tousson,77,0,1.0,...,48.346979,2020,2020-S2,2020-T3,2020-S1,2019,2020T2,1142.857143,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
815669,2025-06-23,Vente,835000.0,RUE SAINT MAUR,75011.0,75111,Paris 11e Arrondissement,75,2,2.0,...,48.869697,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429,2024-S2,9848.571429
815670,2025-06-25,Vente,375000.0,RUE FABRE D EGLANTINE,75012.0,75112,Paris 12e Arrondissement,75,2,2.0,...,48.846075,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364,2024-S2,8888.888889
815671,2025-06-25,Vente,1370000.0,AV PAUL DOUMER,75016.0,75116,Paris 16e Arrondissement,75,1,2.0,...,48.861611,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348,2024-S2,10500.000000
815672,2025-06-27,Vente,550000.0,RUE VERGNIAUD,75013.0,75113,Paris 13e Arrondissement,75,2,2.0,...,48.823258,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443,2024-S2,8792.452830


In [ ]:
raw_idf_data = raw_idf_data.drop(columns="semestre_y")
raw_idf_data = raw_idf_data.rename(columns={"commune_prix_median_m2": "commune_prix_median_m2_semestre_prec",
                                            "semestre_x" : "semestre"})
raw_idf_data.isnull().mean() * 100

date_mutation                            0.000000
nature_mutation                          0.000000
valeur_fonciere                          0.000000
adresse_nom_voie                         0.000368
code_postal                              0.000981
code_commune                             0.000000
nom_commune                              0.000000
code_departement                         0.000000
nombre_lots                              0.000000
code_type_local                          0.000000
type_local                               0.000000
surface_reelle_bati                      0.000000
nombre_pieces_principales                0.000000
surface_terrain                         68.927905
longitude                                1.440649
latitude                                 1.440649
annee                                    0.000000
semestre                                 0.000000
trimestre                                0.000000
semestre_precedent                       0.000000


On a 12% des ventes qui n'ont pas de référence de prix médian de la commune. Il y a forcément toutes les ventes du 2ème semestre de 2020 car c'est le début de l'historique. On va supprimer toutes ces ventes.  

In [ ]:
raw_idf_data = raw_idf_data[raw_idf_data["date_mutation"] >= "2021-01-01"]

raw_idf_data.isnull().mean() * 100

date_mutation                            0.000000
nature_mutation                          0.000000
valeur_fonciere                          0.000000
adresse_nom_voie                         0.000419
code_postal                              0.001117
code_commune                             0.000000
nom_commune                              0.000000
code_departement                         0.000000
nombre_lots                              0.000000
code_type_local                          0.000000
type_local                               0.000000
surface_reelle_bati                      0.000000
nombre_pieces_principales                0.000000
surface_terrain                         69.191419
longitude                                1.346308
latitude                                 1.346308
annee                                    0.000000
semestre                                 0.000000
trimestre                                0.000000
semestre_precedent                       0.000000


Il y a desormais très peu de valeurs manquantes sur les variables prix moyen/médian au mètre carré (seules les petites communes n'observent pas de ventes au semestre précedent)

Calculer l'écart du prix du bien par rapport au prix médian de la commune 


In [ ]:
raw_idf_data['ecart_prix_median_pct'] = (
    (raw_idf_data['prix_m2'] - raw_idf_data['commune_prix_median_m2_semestre_prec']) /
    raw_idf_data['commune_prix_median_m2_semestre_prec'] * 100
)

raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,...,latitude,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,commune_prix_median_m2_semestre_prec,ecart_prix_median_pct
99419,2021-01-05,Vente,352000.0,RUE DE L EGLISE,77115.0,77453,Sivry-Courtry,77,0,1.0,...,48.527196,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2752.702703,6.561937
99420,2021-01-05,Vente,352000.0,RUE DE L EGLISE,77115.0,77453,Sivry-Courtry,77,0,1.0,...,48.527196,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2752.702703,6.561937
99421,2021-01-05,Vente,81000.0,RUE DES BASSES LOGES,77210.0,77014,Avon,77,2,2.0,...,48.416896,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,1421.052632,2941.176471,-51.684211
99422,2021-01-08,Vente,165000.0,RUE DU MOULIN A VENT,77127.0,77251,Lieusaint,77,1,2.0,...,48.628821,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2500.000000,3225.000000,-22.480620
99423,2021-01-08,Vente,235000.0,RUE MIRABEAU,77330.0,77350,Ozoir-la-Ferrière,77,0,1.0,...,48.768597,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5731.707317,3857.575758,48.583143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
815669,2025-06-23,Vente,835000.0,RUE SAINT MAUR,75011.0,75111,Paris 11e Arrondissement,75,2,2.0,...,48.869697,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429,9848.571429,21.119814
815670,2025-06-25,Vente,375000.0,RUE FABRE D EGLANTINE,75012.0,75112,Paris 12e Arrondissement,75,2,2.0,...,48.846075,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364,8888.888889,27.840909
815671,2025-06-25,Vente,1370000.0,AV PAUL DOUMER,75016.0,75116,Paris 16e Arrondissement,75,1,2.0,...,48.861611,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348,10500.000000,41.821946
815672,2025-06-27,Vente,550000.0,RUE VERGNIAUD,75013.0,75113,Paris 13e Arrondissement,75,2,2.0,...,48.823258,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443,8792.452830,2.546964


In [ ]:
raw_idf_data['nature_mutation'].unique()

array(['Vente', "Vente en l'état futur d'achèvement", 'Adjudication',
       'Echange', 'Vente terrain à bâtir', 'Expropriation'], dtype=object)

On ne garde que les ventes 

In [ ]:
raw_idf_data = raw_idf_data[raw_idf_data['nature_mutation'].isin(['Vente', "Vente en l'état futur d'achèvement"])].copy()
raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,...,latitude,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,commune_prix_median_m2_semestre_prec,ecart_prix_median_pct
99419,2021-01-05,Vente,352000.0,RUE DE L EGLISE,77115.0,77453,Sivry-Courtry,77,0,1.0,...,48.527196,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2752.702703,6.561937
99420,2021-01-05,Vente,352000.0,RUE DE L EGLISE,77115.0,77453,Sivry-Courtry,77,0,1.0,...,48.527196,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2933.333333,2752.702703,6.561937
99421,2021-01-05,Vente,81000.0,RUE DES BASSES LOGES,77210.0,77014,Avon,77,2,2.0,...,48.416896,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,1421.052632,2941.176471,-51.684211
99422,2021-01-08,Vente,165000.0,RUE DU MOULIN A VENT,77127.0,77251,Lieusaint,77,1,2.0,...,48.628821,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,2500.000000,3225.000000,-22.480620
99423,2021-01-08,Vente,235000.0,RUE MIRABEAU,77330.0,77350,Ozoir-la-Ferrière,77,0,1.0,...,48.768597,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5731.707317,3857.575758,48.583143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
815669,2025-06-23,Vente,835000.0,RUE SAINT MAUR,75011.0,75111,Paris 11e Arrondissement,75,2,2.0,...,48.869697,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11928.571429,9848.571429,21.119814
815670,2025-06-25,Vente,375000.0,RUE FABRE D EGLANTINE,75012.0,75112,Paris 12e Arrondissement,75,2,2.0,...,48.846075,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11363.636364,8888.888889,27.840909
815671,2025-06-25,Vente,1370000.0,AV PAUL DOUMER,75016.0,75116,Paris 16e Arrondissement,75,1,2.0,...,48.861611,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,14891.304348,10500.000000,41.821946
815672,2025-06-27,Vente,550000.0,RUE VERGNIAUD,75013.0,75113,Paris 13e Arrondissement,75,2,2.0,...,48.823258,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9016.393443,8792.452830,2.546964


In [ ]:
# Trier et réinitialiser l'index
raw_idf_data.sort_values('date_mutation', inplace=True)
raw_idf_data.reset_index(drop=True, inplace=True)

print(f"Dataset de valeurs foncières : {raw_idf_data.shape}")

Dataset de valeurs foncières : (712545, 25)


# 3. Merge entre la table des valeurs foncières et les indicateurs communaux

In [ ]:
data_par_commune


,code_commune,commune_taux_croissance_pop_annuel_2016_2019,commune_taux_croissance_pop_annuel_2016_2020,commune_taux_croissance_pop_annuel_2016_2021,commune_taux_croissance_pop_annuel_2016_2022,commune_taux_croissance_pop_annuel_2016_2023,commune_taux_croissance_pop_annuel_2016_2024,commune_revenu_median_2020,commune_etablissements_agriculture_2021,commune_etablissements_industrie_2021,...,commune_taux_criminalite_moyen,commune_densite_pop_2016,commune_densite_pop_2017,commune_densite_pop_2018,commune_densite_pop_2019,commune_densite_pop_2020,commune_densite_pop_2021,commune_densite_pop_2022,commune_densite_pop_2023,commune_densite_pop_2024
0,78638,0.752379,0.924149,0.746816,0.895685,0.795949,0.853214,28210,0,3,...,0.258535,5.620118e+08,5.579882e+08,5.666272e+08,5.747929e+08,5.830769e+08,5.833136e+08,5.928994e+08,5.940828e+08,6.015385e+08
1,77295,1.425548,1.232726,1.028839,0.905193,0.692137,0.532217,28610,1,1,...,0.053883,1.506881e+08,1.528670e+08,1.550459e+08,1.572248e+08,1.582569e+08,1.586009e+08,1.590596e+08,1.581422e+08,1.572248e+08
2,95301,-2.498214,-1.967614,-1.295598,-0.848355,-0.530436,-0.464286,24820,0,0,...,0.000000,1.171206e+08,1.128405e+08,1.085603e+08,1.085603e+08,1.081712e+08,1.097276e+08,1.112840e+08,1.128405e+08,1.128405e+08
3,78402,0.275029,0.342964,0.442834,0.260297,0.580753,0.673379,26560,2,3,...,0.180551,3.479846e+08,3.500000e+08,3.489443e+08,3.508637e+08,3.527831e+08,3.557582e+08,3.534549e+08,3.623800e+08,3.671785e+08
4,91578,-2.195070,-2.549868,-2.584136,-2.619680,-2.759919,-2.328407,35220,1,0,...,0.000000,7.459954e+07,7.505721e+07,7.254005e+07,6.979405e+07,6.727689e+07,6.544622e+07,6.361556e+07,6.132723e+07,6.178490e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1282,77181,4.878113,5.293295,4.514491,5.117089,4.480795,4.063079,28250,1,12,...,0.386791,4.137778e+08,4.149630e+08,4.462222e+08,4.773333e+08,5.085926e+08,5.160000e+08,5.582222e+08,5.623704e+08,5.690370e+08
1283,95446,0.424359,0.605921,0.276400,0.037023,-0.151565,0.027766,29650,3,4,...,0.157489,1.336553e+08,1.322437e+08,1.338039e+08,1.353640e+08,1.369242e+08,1.355126e+08,1.339525e+08,1.322437e+08,1.339525e+08
1284,95056,0.967752,0.863457,0.891754,0.681734,0.675128,0.652740,27310,2,7,...,0.223398,2.228662e+08,2.278188e+08,2.281349e+08,2.293994e+08,2.306639e+08,2.329821e+08,2.321391e+08,2.336143e+08,2.347734e+08
1285,91570,-0.318442,-0.374790,-0.091907,0.351715,0.861330,0.835221,22570,1,29,...,0.384172,3.791493e+09,3.761059e+09,3.810964e+09,3.755388e+09,3.734972e+09,3.774102e+09,3.872212e+09,4.026087e+09,4.052363e+09


In [ ]:
raw_idf_data

,date_mutation,nature_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,...,latitude,annee,semestre,trimestre,semestre_precedent,annee_precedente,trimestre_precedent,prix_m2,commune_prix_median_m2_semestre_prec,ecart_prix_median_pct
0,2021-01-01,Vente,169500.0,ALL HOCHE,92130.0,92040,Issy-les-Moulineaux,92,2,2.0,...,48.823551,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,9416.666667,8829.268293,6.652855
1,2021-01-02,Vente,512000.0,RUE DE CHARONNE,75011.0,75111,Paris 11e Arrondissement,75,1,2.0,...,48.854636,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,17066.666667,11192.253021,52.486426
2,2021-01-02,Vente,185000.0,AV GAL LECLERC,95250.0,95051,Beauchamp,95,1,2.0,...,49.009098,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,4404.761905,4161.984291,5.833218
3,2021-01-02,Vente,415000.0,RUE DES COURLIS,95100.0,95018,Argenteuil,95,0,1.0,...,48.944657,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5187.500000,3756.720430,38.085868
4,2021-01-02,Vente,415000.0,RUE DES COURLIS,95100.0,95018,Argenteuil,95,0,1.0,...,48.944657,2021,2021-S1,2021-T1,2020-S2,2020,2020T4,5187.500000,3756.720430,38.085868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
712540,2025-06-30,Vente,1771150.0,RUE GARNIER,92200.0,92051,Neuilly-sur-Seine,92,2,2.0,...,48.885924,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,12048.639456,10357.142857,16.331691
712541,2025-06-30,Vente,360000.0,BD DU GENERAL LECLERC,92200.0,92051,Neuilly-sur-Seine,92,2,2.0,...,48.888503,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,11250.000000,10357.142857,8.620690
712542,2025-06-30,Vente,235000.0,RUE PAUL DEROULEDE,92270.0,92009,Bois-Colombes,92,2,2.0,...,48.913087,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,6184.210526,6428.571429,-3.801170
712543,2025-06-30,Vente,562500.0,RUE DE CHEZY,92200.0,92051,Neuilly-sur-Seine,92,3,2.0,...,48.885456,2025,2025-S1,2025-T2,2024-S2,2024,2025T1,9698.275862,10357.142857,-6.361474


In [ ]:
data_par_commune.columns

Index(['code_commune', 'commune_taux_croissance_pop_annuel_2016_2019',
       'commune_taux_croissance_pop_annuel_2016_2020',
       'commune_taux_croissance_pop_annuel_2016_2021',
       'commune_taux_croissance_pop_annuel_2016_2022',
       'commune_taux_croissance_pop_annuel_2016_2023',
       'commune_taux_croissance_pop_annuel_2016_2024',
       'commune_revenu_median_2020', 'commune_etablissements_agriculture_2021',
       'commune_etablissements_industrie_2021',
       'commune_etablissements_construction_2021',
       'commune_etablissements_1_salarie_2021',
       'commune_etablissements_10_plus_salaries_2021',
       'commune_densite_population_active_2020',
       'commune_taux_residences_secondaires', 'commune_taux_logements_vacants',
       'commune_taux_proprietaires',
       'commune_ratio_residences_secondaires_population',
       'commune_densite_residences_principales_2020',
       'commune_taux_chomage_15_64', 'commune_taux_emploi',
       'commune_evolution_emploi_2

Pour éviter au maximum le leakage, on essaye d'utiliser quand on le peut les données de l'**année précédant la vente** :
- Vente en 2021 → données 2020
- Vente en 2024 → données 2023
- etc.

Règles :
1. **Variables avec millésime fixe** (2013, 2014, 2020, 2021) : toujours disponibles même si il y a un leak pour les ventes en 2020, 2021 qui utilisent des indicateurs de la même année. 
2. **Taux de croissance démographique** : on prend celui de l'année précédant la vente
3. **Densité de population** : on prend celle de l'année précédant la vente

In [ ]:
colonnes_statiques = [
    'code_commune',
    'commune_revenu_median_2020',
    'commune_etablissements_agriculture_2021',
    'commune_etablissements_industrie_2021',
    'commune_etablissements_construction_2021',
    'commune_etablissements_1_salarie_2021',
    'commune_etablissements_10_plus_salaries_2021',
    'commune_densite_population_active_2020',
    'commune_taux_residences_secondaires',
    'commune_taux_logements_vacants',
    'commune_taux_proprietaires',
    'commune_ratio_residences_secondaires_population',
    'commune_densite_residences_principales_2020',
    'commune_taux_chomage_15_64',
    'commune_taux_emploi',
    'commune_evolution_emploi_2014_2020',
    'commune_part_services_entreprises',
    'commune_part_commerce_tourisme',
    'commune_part_admin_sante',
    'commune_etablissements_par_menage',
    'commune_taux_etablissements_10_plus',
    'commune_sante_score_2013_commune',
    'commune_education_score_2013_commune',
    'commune_revenu_score_2013_commune',
    'commune_idh2_2013_commune',
    'commune_taux_criminalite_moyen'
]

print(f"Shape avant merge statique : {raw_idf_data.shape}")


raw_idf_data = raw_idf_data.merge(
    data_par_commune[colonnes_statiques],
    on='code_commune',
    how='left'
)

print(f"Shape après merge statique : {raw_idf_data.shape}")

Shape avant merge statique : (712545, 25)
Shape après merge statique : (712545, 50)
Shape après merge statique : (712545, 50)


In [ ]:
# Mapping année de référence -> taux de croissance à utiliser
# Pour une vente en 2021 , on prend le taux 2016-2020
mapping_taux_croissance = {
    2020: 'commune_taux_croissance_pop_annuel_2016_2019',
    2021: 'commune_taux_croissance_pop_annuel_2016_2020',
    2022: 'commune_taux_croissance_pop_annuel_2016_2021',
    2023: 'commune_taux_croissance_pop_annuel_2016_2022',
    2024: 'commune_taux_croissance_pop_annuel_2016_2023'
}

# Mapping année de référence -> densité de population à utiliser
# Pour une vente en 2021, on prend la densité 2020
mapping_densite = {
    2020: 'commune_densite_pop_2019',
    2021: 'commune_densite_pop_2020',
    2022: 'commune_densite_pop_2021',
    2023: 'commune_densite_pop_2022',
    2024: 'commune_densite_pop_2023'
}

# On va créer un mapping code_commune + année_reference -> valeurs
# Préparer la table de lookup pour les taux de croissance et densités
taux_croissance_cols = list(mapping_taux_croissance.values())
densite_cols = list(mapping_densite.values())

# Créer une table longue avec code_commune et année_reference
lookup_data = []

for code_commune in data_par_commune['code_commune'].unique():
    commune_row = data_par_commune[data_par_commune['code_commune'] == code_commune].iloc[0]
    
    # Pour chaque année de référence possible
    for annee, col_taux in mapping_taux_croissance.items():
        for annee_dens, col_dens in mapping_densite.items():
            if annee == annee_dens:  # Même année de référence
                lookup_data.append({
                    'code_commune': code_commune,
                    'annee_vente': annee,
                    'commune_taux_croissance_pop': commune_row.get(col_taux, np.nan),
                    'commune_densite_pop': commune_row.get(col_dens, np.nan)
                })

lookup_table = pd.DataFrame(lookup_data)
print(f"Table de lookup créée : {lookup_table.shape}")
lookup_table.head(10)

Table de lookup créée : (6435, 4)


,code_commune,annee,commune_taux_croissance_pop,commune_densite_pop
0,78638,2020,0.752379,5.747929e+08
1,78638,2021,0.924149,5.830769e+08
2,78638,2022,0.746816,5.833136e+08
3,78638,2023,0.895685,5.928994e+08
4,78638,2024,0.795949,5.940828e+08
5,77295,2020,1.425548,1.572248e+08
6,77295,2021,1.232726,1.582569e+08
7,77295,2022,1.028839,1.586009e+08
8,77295,2023,0.905193,1.590596e+08
9,77295,2024,0.692137,1.581422e+08


In [ ]:
raw_idf_data = raw_idf_data.merge(
    lookup_table,
    left_on=['code_commune', 'annee'],
    right_on=['code_commune', 'annee_vente'],
    how='left'
).drop(columns=['annee_vente'])


print(f"Shape finale après merge temporel : {raw_idf_data.shape}")

Shape finale après merge temporel : (712545, 53)


In [ ]:
raw_idf_data.isnull().mean() * 100

date_mutation                                       0.000000
nature_mutation                                     0.000000
valeur_fonciere                                     0.000000
adresse_nom_voie                                    0.000421
code_postal                                         0.001123
code_commune                                        0.000000
nom_commune                                         0.000000
code_departement                                    0.000000
nombre_lots                                         0.000000
code_type_local                                     0.000000
type_local                                          0.000000
surface_reelle_bati                                 0.000000
nombre_pieces_principales                           0.000000
surface_terrain                                    69.199419
longitude                                           1.341670
latitude                                            1.341670
annee_x                 

In [ ]:
# Sauvegarder le dataset final
output_path = "data/processed/idf_vf_full.parquet"
raw_idf_data.to_parquet(output_path, index=False)

print(f"✓ Dataset final sauvegardé : {output_path}")
print(f"  - Shape : {raw_idf_data.shape}")
print(f"  - Période : {raw_idf_data['date_mutation'].min()} à {raw_idf_data['date_mutation'].max()}")
print(f"  - Nombre de communes : {raw_idf_data['code_commune'].nunique()}")